# CNN Proof of Concept

Train and test a simple CNN on synthetic bitstream images.

## Goals

1. Create synthetic dataset (clean + Trojan)
2. Train BitstreamCNN
3. Evaluate performance
4. Visualize learned features

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

from ml.models.cnn_bitstream import BitstreamCNN

## Generate synthetic dataset

In [ ]:
def generate_synthetic_bitstream(width=256, height=256, has_trojan=False):
    """
    Generate a synthetic bitstream image.
    
    Clean: Low entropy background
    Trojan: High entropy patch in center
    """
    # Base: low entropy (mostly 0s with some structure)
    img = np.zeros((height, width), dtype=np.float32)
    
    # Add some periodic structure (simulates FPGA columns)
    for col in range(0, width, 8):
        img[:, col] = 0.5
    
    # Add noise
    img += 0.1 * np.random.randn(height, width).astype(np.float32)
    
    if has_trojan:
        # Insert high-entropy patch (simulated Trojan)
        h_start = height // 2 - 25
        h_end = height // 2 + 25
        w_start = width // 2 - 25
        w_end = width // 2 + 25
        img[h_start:h_end, w_start:w_end] = np.random.rand(50, 50).astype(np.float32)
    
    # Clip to [0, 1]
    img = np.clip(img, 0, 1)
    
    return img

# Generate dataset
num_clean = 100
num_trojan = 100

X_train = []
y_train = []

for i in range(num_clean):
    img = generate_synthetic_bitstream(has_trojan=False)
    X_train.append(img[None, :, :])  # Add channel dimension
    y_train.append(0)  # Clean

for i in range(num_trojan):
    img = generate_synthetic_bitstream(has_trojan=True)
    X_train.append(img[None, :, :])
    y_train.append(1)  # Trojan

X_train = torch.tensor(np.array(X_train), dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

print(f"Dataset shape: {X_train.shape}")
print(f"Labels shape: {y_train.shape}")

## Visualize samples

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i in range(4):
    # Clean samples
    axes[0, i].imshow(X_train[i, 0], cmap='gray')
    axes[0, i].set_title(f'Clean {i+1}')
    axes[0, i].axis('off')
    
    # Trojan samples
    axes[1, i].imshow(X_train[num_clean + i, 0], cmap='gray')
    axes[1, i].set_title(f'Trojan {i+1}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## Train CNN

In [ ]:
# Create DataLoader
dataset = TensorDataset(X_train, y_train)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BitstreamCNN(num_classes=2).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop
epochs = 20
train_losses = []
train_accs = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    
    print(f"Epoch {epoch+1}/{epochs}: loss={epoch_loss:.4f}, acc={epoch_acc:.4f}")

## Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True)

axes[1].plot(train_accs)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Evaluate on test samples

In [ ]:
# Generate test set
X_test = []
y_test = []

for i in range(20):
    img = generate_synthetic_bitstream(has_trojan=False)
    X_test.append(img[None, :, :])
    y_test.append(0)

for i in range(20):
    img = generate_synthetic_bitstream(has_trojan=True)
    X_test.append(img[None, :, :])
    y_test.append(1)

X_test = torch.tensor(np.array(X_test), dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.long).to(device)

# Evaluate
model.eval()
with torch.no_grad():
    logits = model(X_test)
    preds = logits.argmax(dim=1)
    accuracy = (preds == y_test).float().mean().item()

print(f"Test accuracy: {accuracy:.4f}")

## Next Steps

- Test on real bitstream data from Quartus
- Experiment with different architectures (deeper, ResNet, attention)
- Add data augmentation
- Implement cross-validation